In [149]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import tree
from sklearn import ensemble
import xgboost
np.random.seed(7)

In [2]:
data = pd.read_excel("career_dataset_large.xlsx")

In [3]:
data.head()

,Education Level,Specialization,Skills,Certifications,CGPA/Percentage,Recommended Career
0,Bachelor's,Finance,"Counseling, MS Office, Machine Learning",Tally ERP,67,Business Analyst
1,Intermediate,Science,"Accounting, MS Office",AWS Certified,67,Software Engineer
2,Master's,Business,"Accounting, SQL, Data Analysis",Mental Health Basics,90,Financial Analyst
3,Bachelor's,Computer Science,Communication,NaN,75,Clerk
4,Matric,Business,Data Analysis,Tally ERP,83,Sales Assistant


In [4]:
data.describe()

,CGPA/Percentage
count,5000.000000
mean,77.479800
std,10.396288
min,60.000000
25%,68.000000
50%,77.000000
75%,87.000000
max,95.000000


In [39]:
columns_to_binarize = [ 0, 1, 2, 3]
binary_to_index = {}

for column_index in columns_to_binarize:
    for value in data.iloc[:, column_index]:
        if pd.notna(value):
            for splits in str(value).split(', '):
                binary_to_index[splits] = column_index
binary_to_index

{"Bachelor's": 0,
 'Intermediate': 0,
 "Master's": 0,
 'Matric': 0,
 'PhD': 0,
 'Finance': 1,
 'Science': 1,
 'Business': 1,
 'Computer Science': 1,
 'Arts': 1,
 'Psychology': 1,
 'Commerce': 1,
 'Engineering': 1,
 'Counseling': 2,
 'MS Office': 2,
 'Machine Learning': 2,
 'Accounting': 2,
 'SQL': 2,
 'Data Analysis': 2,
 'Communication': 2,
 'Financial Analysis': 2,
 'Python': 2,
 'Marketing': 2,
 'Tally ERP': 3,
 'AWS Certified': 3,
 'Mental Health Basics': 3,
 'Digital Marketing': 3,
 'CFA Level 1': 3,
 'Creative Writing': 3,
 'Google Data Analytics': 3}

In [40]:
feature_names = list(binary_to_index.keys()) + ['CGPA/Percentage']
feature_names

["Bachelor's",
 'Intermediate',
 "Master's",
 'Matric',
 'PhD',
 'Finance',
 'Science',
 'Business',
 'Computer Science',
 'Arts',
 'Psychology',
 'Commerce',
 'Engineering',
 'Counseling',
 'MS Office',
 'Machine Learning',
 'Accounting',
 'SQL',
 'Data Analysis',
 'Communication',
 'Financial Analysis',
 'Python',
 'Marketing',
 'Tally ERP',
 'AWS Certified',
 'Mental Health Basics',
 'Digital Marketing',
 'CFA Level 1',
 'Creative Writing',
 'Google Data Analytics',
 'CGPA/Percentage']

In [41]:
target = 'Recommended Career'

In [42]:
x = pd.DataFrame(np.zeros( (len(data), len(feature_names)), dtype=int), columns = feature_names)

In [43]:
for index in range(len(feature_names)):
    feature_name = feature_names[index]
    if feature_name not in binary_to_index:
        x[feature_name] = data[feature_name]
        continue
        
    column_index = binary_to_index[feature_name]

    for row in range(len(data)):
        value = data.iloc[row, column_index]
        if pd.notna(value):
            split = value.split(', ')
            x.iloc[row, index] = int(feature_name in split)

x

,Bachelor's,Intermediate,Master's,Matric,PhD,Finance,Science,Business,Computer Science,Arts,...,Python,Marketing,Tally ERP,AWS Certified,Mental Health Basics,Digital Marketing,CFA Level 1,Creative Writing,Google Data Analytics,CGPA/Percentage
0,1,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,67
1,0,1,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,67
2,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,90
3,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,75
4,0,0,0,1,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0,0,0,1,0,0,0,0,0,0,...,1,0,1,0,0,0,0,0,0,90
4996,0,0,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,1,0,92
4997,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,63
4998,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,83


In [44]:
target_names = data[target].unique()

y = data[target]

label_encoder = preprocessing.LabelEncoder()
y_encoded = label_encoder.fit_transform( data[target] )

y_encoded

array([ 0, 11,  3, ...,  3, 11,  1], shape=(5000,))

In [45]:
x_train, x_test, y_train, y_test = model_selection.train_test_split(x, y_encoded, test_size=0.33, stratify=y, random_state = 7)
scaler = preprocessing.StandardScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)

In [97]:
number_of_estimators = np.arange(10, 110, 10)
number_of_estimators

array([ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100])

In [98]:
max_depths = np.arange(5, 55, 5)
max_depths

array([ 5, 10, 15, 20, 25, 30, 35, 40, 45, 50])

In [99]:
def train_and_evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)

    return model.score(x_test, y_test)

In [111]:
optimal_number_of_estimators = number_of_estimators[0]
optimal_depth = max_depths[0]
optimal_accuracy = 0

for num in number_of_estimators:
    for depth in max_depths:
        model = ensemble.RandomForestClassifier(num, max_depth=depth, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_test, y_test)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth)

100 5


In [112]:
optimal_accuracy

0.09212121212121212

In [122]:
number_of_estimators = np.arange(90, 110, 2)
number_of_estimators

max_depths = np.arange(3, 13, 1)
max_depths

for num in number_of_estimators:
    for depth in max_depths:
        model = ensemble.RandomForestClassifier(num, max_depth=depth, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_test, y_test)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

102 5 0.09393939393939393


In [123]:
model = ensemble.RandomForestClassifier(optimal_number_of_estimators, max_depth=optimal_depth, random_state=7)

In [124]:
model.fit(x_train, y_train)

,n_estimators,np.int64(102)
,criterion,'gini'
,max_depth,np.int64(5)
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [125]:
y_test_predicted = model.predict(x_test)
y_test_predicted

array([5, 1, 1, ..., 1, 3, 6], shape=(1650,))

In [126]:
y_test_predicted_strings = label_encoder.inverse_transform(y_test_predicted)
y_test_predicted_strings

array(['ML Engineer', 'Clerk', 'Clerk', ..., 'Clerk', 'Financial Analyst',
       'Marketing Executive'], shape=(1650,), dtype=object)

In [127]:
model.score(x_train, y_train)

0.36388059701492537

In [128]:
model.score(x_test, y_test)

0.09393939393939393

In [129]:
metrics.confusion_matrix(y_test, y_test_predicted)

array([[ 1, 25, 14,  8,  6, 11, 22,  6, 11,  7, 12,  7],
       [ 4, 34,  7,  6,  4, 12, 26, 12, 11,  5, 16,  6],
       [ 6, 28, 15,  7,  5, 12, 23,  1, 15,  4, 14,  6],
       [ 5, 13, 17, 10,  6, 14, 24,  7,  8,  7, 16,  9],
       [ 4, 26, 11, 13,  5, 14, 26,  7, 12,  1, 12,  5],
       [ 4, 32, 10,  6, 10, 18, 15,  3, 14,  4, 20,  6],
       [ 4, 18, 14,  4,  6, 28, 26,  6,  8,  9,  8, 12],
       [ 2, 20, 10,  6,  9, 12, 28,  9, 14,  8,  9,  7],
       [ 4, 22,  6, 10,  8, 12, 27,  9, 15,  7, 11,  7],
       [ 3, 14, 20,  8,  3, 11, 27,  5, 14,  5, 16, 10],
       [ 4, 20, 12,  2,  3, 19, 25,  9, 17, 10, 12,  5],
       [ 1, 20, 13,  8,  8, 15, 20, 11, 18,  7, 12,  5]])

In [130]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.02      0.01      0.01       130
           1       0.12      0.24      0.16       143
           2       0.10      0.11      0.11       136
           3       0.11      0.07      0.09       136
           4       0.07      0.04      0.05       136
           5       0.10      0.13      0.11       142
           6       0.09      0.18      0.12       143
           7       0.11      0.07      0.08       134
           8       0.10      0.11      0.10       138
           9       0.07      0.04      0.05       136
          10       0.08      0.09      0.08       138
          11       0.06      0.04      0.04       138

    accuracy                           0.09      1650
   macro avg       0.09      0.09      0.08      1650
weighted avg       0.09      0.09      0.08      1650



In [131]:
metrics.accuracy_score(y_test, y_test_predicted)

0.09393939393939393

In [188]:
number_of_estimators = np.arange(50, 150, 10)
number_of_estimators

max_depths = np.arange(5, 25, 2)
max_depths

optimal_number_of_estimators = number_of_estimators[0]
optimal_depth = max_depths[0]
optimal_accuracy = 0

for num in number_of_estimators:
    for depth in max_depths:
        model = ensemble.AdaBoostClassifier(estimator=tree.DecisionTreeClassifier(max_depth=depth), n_estimators=num, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_test, y_test)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

60 5 0.09636363636363636


In [189]:
model = ensemble.AdaBoostClassifier(estimator=tree.DecisionTreeClassifier(max_depth=optimal_depth), n_estimators=optimal_number_of_estimators, random_state=7)

In [190]:
model.fit(x_train, y_train)

,estimator,DecisionTreeC...h=np.int64(5))
,n_estimators,np.int64(60)
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,7
,criterion,'gini'
,splitter,'best'
,max_depth,np.int64(5)
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [191]:
y_test_predicted = model.predict(x_test)
y_test_predicted

array([7, 3, 1, ..., 0, 3, 6], shape=(1650,))

In [192]:
model.score(x_train, y_train)

0.282089552238806

In [193]:
model.score(x_test, y_test)

0.09636363636363636

In [194]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.10      0.07      0.08       130
           1       0.10      0.13      0.11       143
           2       0.05      0.06      0.06       136
           3       0.14      0.11      0.12       136
           4       0.10      0.10      0.10       136
           5       0.15      0.12      0.13       142
           6       0.14      0.18      0.16       143
           7       0.06      0.08      0.07       134
           8       0.10      0.11      0.11       138
           9       0.08      0.04      0.06       136
          10       0.06      0.06      0.06       138
          11       0.08      0.09      0.08       138

    accuracy                           0.10      1650
   macro avg       0.10      0.10      0.09      1650
weighted avg       0.10      0.10      0.10      1650



In [195]:
metrics.confusion_matrix(y_test, y_test_predicted)

array([[ 9, 13,  6, 10, 14,  5, 17, 17, 16,  3, 11,  9],
       [ 4, 18, 11,  8, 12,  9, 16, 20, 14,  6, 13, 12],
       [14, 18,  8,  9,  9, 10, 10, 12, 11,  6, 16, 13],
       [ 6, 17,  9, 15, 12, 13, 11, 14, 11,  5,  9, 14],
       [ 8, 16, 23,  9, 14, 10, 15,  3, 12,  3,  9, 14],
       [ 4, 19, 15,  4,  9, 17, 13, 17, 10,  8, 11, 15],
       [ 5, 18, 15,  7, 12,  8, 26,  9,  9, 10, 12, 12],
       [12, 14,  9,  8, 16,  4, 22, 11,  8, 10,  8, 12],
       [ 7, 10, 14, 14,  6, 12, 11, 13, 15,  6, 16, 14],
       [ 9,  9, 17,  8, 12, 12, 17, 17, 10,  6, 11,  8],
       [10, 15, 13,  3, 12,  5, 20, 17, 15,  8,  8, 12],
       [ 2, 19, 14,  9, 15,  8, 12, 21, 13,  4,  9, 12]])

In [177]:
number_of_estimators = np.arange(50, 150, 10)
number_of_estimators

max_depths = np.arange(5, 25, 2)
max_depths

optimal_number_of_estimators = number_of_estimators[0]
optimal_depth = max_depths[0]
optimal_accuracy = 0

for num in number_of_estimators:
    for depth in max_depths:
        model = xgboost.XGBClassifier(n_estimators=num, max_depth=depth, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_test, y_test)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

80 7 0.08606060606060606


In [178]:
model = xgboost.XGBClassifier(n_estimators=optimal_number_of_estimators, max_depth=optimal_depth, random_state=7)

In [179]:
model.fit(x_train, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [180]:
y_test_predicted = model.predict(x_test)
y_test_predicted

array([5, 1, 1, ..., 0, 3, 7], shape=(1650,))

In [181]:
model.score(x_train, y_train)

0.9916417910447761

In [182]:
model.score(x_test, y_test)

0.08606060606060606

In [185]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.10      0.08      0.09       130
           1       0.09      0.12      0.10       143
           2       0.06      0.07      0.07       136
           3       0.11      0.09      0.10       136
           4       0.07      0.06      0.07       136
           5       0.13      0.13      0.13       142
           6       0.09      0.09      0.09       143
           7       0.04      0.04      0.04       134
           8       0.09      0.09      0.09       138
           9       0.06      0.06      0.06       136
          10       0.11      0.12      0.11       138
          11       0.07      0.07      0.07       138

    accuracy                           0.09      1650
   macro avg       0.09      0.09      0.09      1650
weighted avg       0.09      0.09      0.09      1650



In [186]:
metrics.confusion_matrix(y_test, y_test_predicted)

array([[11, 12,  8,  7, 12,  8, 10, 15, 18,  9, 10, 10],
       [14, 17,  9,  7, 10, 13, 13, 14, 15, 12, 11,  8],
       [10, 19, 10,  9,  8, 16, 11,  9, 12,  5, 17, 10],
       [12, 10, 11, 12,  8, 10, 18, 11,  9, 11, 16,  8],
       [ 4, 18, 22, 13,  8, 10, 10,  7,  9, 14,  8, 13],
       [ 4, 14, 15, 10,  6, 19,  5, 19, 13,  6, 19, 12],
       [ 8, 16, 22,  8, 10,  8, 13, 11, 11, 12, 11, 13],
       [12, 14, 10,  6, 11, 15, 11,  6, 11, 15,  9, 14],
       [ 7, 12,  9, 12, 11, 13, 13, 10, 13, 14, 10, 14],
       [10, 14, 14,  6,  7, 11, 19, 11, 13,  8, 12, 11],
       [ 9, 16, 14, 10,  6,  9,  9, 13, 13, 12, 16, 11],
       [10, 19, 11, 13, 11, 11, 10, 15, 13,  7,  9,  9]])